In [0]:
USE CATALOG credit_card_project;

-- Step 1: Create the groups conceptually (done via Databricks Account Console UI in practice,
-- or via SCIM/Azure AD sync — shown here as the group names the masking functions check)
-- Groups: 'risk_analysts' (full visibility) and 'marketing_analysts' (masked)

-- Step 2: Masking function for names
CREATE OR REPLACE FUNCTION gold.mask_name(value STRING)
RETURNS STRING
RETURN CASE
  WHEN is_account_group_member('risk_analysts') THEN value
  ELSE 'REDACTED'
END;

-- Step 3: Masking function for email
CREATE OR REPLACE FUNCTION gold.mask_email(value STRING)
RETURNS STRING
RETURN CASE
  WHEN is_account_group_member('risk_analysts') THEN value
  ELSE CONCAT('***@', SPLIT(value, '@')[1])   -- e.g. ***@email.com
END;

-- Step 4: Masking function for phone
CREATE OR REPLACE FUNCTION gold.mask_phone(value STRING)
RETURNS STRING
RETURN CASE
  WHEN is_account_group_member('risk_analysts') THEN value
  ELSE CONCAT('XXXXXX', RIGHT(value, 4))     -- show only last 4 digits
END;

-- Step 5: Attach the masks to the actual columns
ALTER TABLE gold.dim_customer ALTER COLUMN First_Name SET MASK gold.mask_name;
ALTER TABLE gold.dim_customer ALTER COLUMN Last_Name  SET MASK gold.mask_name;
ALTER TABLE gold.dim_customer ALTER COLUMN Email      SET MASK gold.mask_email;
ALTER TABLE gold.dim_customer ALTER COLUMN Phone      SET MASK gold.mask_phone;

-- Step 6: Grant table access to both groups (masking then determines what each sees)
GRANT SELECT ON TABLE gold.dim_customer TO risk_analysts;
GRANT SELECT ON TABLE gold.dim_customer TO marketing_analysts;